# Topic A — Kiểm tra calibration LiDAR–camera

**Đoàn Quang Thắng · 2A202602395 · Track 4 Day21**

Chạy toàn bộ 20 frame KITTI, 80 frame nuScenes và 5 frame synthetic trên CPU.
Thay đổi từng góc/hướng dịch chuyển riêng; đo FOV, độ lệch pixel, điểm trong box
và thử score khoảng cách tới biên Canny. Mọi bảng và ảnh được tạo từ lần chạy này.

Code được đóng gói từ bản địa phương, kiểm tra SHA-256 trước chạy; dữ liệu lấy từ
Git revision cố định. Không cần push Git hoặc tạo Kaggle dataset.
Nguồn ảnh: KITTI Vision Benchmark Suite; nuScenes (Motional), dùng cho học tập.
AI hỗ trợ code/rà soát: Codex qua Herdr; kiểm chứng bằng test và chạy dữ liệu thật.


In [ ]:
import subprocess, sys
DEPENDENCIES = ['numpy==2.2.6', 'opencv-python-headless==4.12.0.88', 'matplotlib==3.10.7', 'pandas==2.3.3']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *DEPENDENCIES], check=True)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, uuid

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
data_checkout = Path('/tmp') / ('day21_data_' + run_id)
run_root = Path('/tmp') / ('day21_code_' + run_id)
run_root.mkdir(exist_ok=False)
REPO = 'https://github.com/conanWinner/DoanQuangThang-2A202602395-Track4-Day21.git'
DATA_REVISION = 'bce73adec3dbd09b2869ed2061513328ee228272'

subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPO, str(data_checkout)], check=True)
subprocess.run(['git', '-C', str(data_checkout), 'checkout', '--quiet', '--detach', DATA_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(data_checkout), 'rev-parse', 'HEAD'], text=True).strip() == DATA_REVISION
(run_root / 'data').symlink_to(data_checkout / 'data', target_is_directory=True)
PAYLOAD = {'starter/__init__.py': '', 'starter/data_health.py': '"""Thống kê sức khoẻ point cloud cho từng frame -> CSV.\n\nMọi học viên nên chạy ở CP0/CP1, trước khi benchmark model ("benchmark data health trước").\nTopic E mở rộng file này thành dashboard (histogram, time gaps, density theo góc...).\n\n    python -m starter.data_health --data-root data/synthetic --out results/data_health.csv\n    python -m starter.data_health --data-root data/kitti_mini --out results/data_health_kitti.csv\n    python -m starter.data_health --data-root data/nuscenes_mini_subset --out results/data_health_nusc.csv\n\nLưu ý nuScenes: trục x của LiDAR hướng sang phải xe, nên azimuth 0° là bên phải, không phải phía trước.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom starter.datasets import list_frames, load_points\n\n\ndef point_stats(points: np.ndarray, n_azimuth_bins: int = 36) -> dict[str, float]:\n    finite = np.isfinite(points).all(axis=1)\n    p = points[finite]\n    rng = np.linalg.norm(p[:, :2], axis=1)\n    az = np.degrees(np.arctan2(p[:, 1], p[:, 0]))\n    az_hist, _ = np.histogram(az, bins=n_azimuth_bins, range=(-180, 180))\n    return {\n        "n_points": int(len(points)),\n        "invalid_ratio": float(1 - finite.mean()) if len(points) else 1.0,\n        "range_min": float(rng.min()) if len(rng) else np.nan,\n        "range_p50": float(np.percentile(rng, 50)) if len(rng) else np.nan,\n        "range_p95": float(np.percentile(rng, 95)) if len(rng) else np.nan,\n        "range_max": float(rng.max()) if len(rng) else np.nan,\n        "ratio_beyond_50m": float((rng > 50).mean()) if len(rng) else np.nan,\n        "z_min": float(p[:, 2].min()) if len(p) else np.nan,\n        "z_max": float(p[:, 2].max()) if len(p) else np.nan,\n        "intensity_mean": float(p[:, 3].mean()) if len(p) else np.nan,\n        "empty_azimuth_bins": int((az_hist == 0).sum()),\n    }\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Thống kê sức khoẻ point cloud của từng frame, ghi ra CSV")\n    ap.add_argument("--data-root", default="data/synthetic", help="thư mục KITTI hoặc nuScenes")\n    ap.add_argument("--out", default="results/data_health.csv", help="file CSV kết quả")\n    args = ap.parse_args()\n\n    rows = [{"frame_id": fid, **point_stats(load_points(args.data_root, fid))} for fid in list_frames(args.data_root)]\n    if not rows:\n        raise SystemExit(f"Không tìm thấy frame nào trong {args.data_root}. Xem data/README.md để tải dữ liệu.")\n\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with out.open("w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\n        w.writeheader()\n        w.writerows(rows)\n    for r in rows:\n        print(f"{r[\'frame_id\']}: n={r[\'n_points\']} invalid={r[\'invalid_ratio\']:.2%} "\n              f"range_p95={r[\'range_p95\']:.1f}m empty_az_bins={r[\'empty_azimuth_bins\']}")\n    print(f"-> {out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'starter/datasets.py': '"""Một cửa duy nhất để đọc dữ liệu: tự nhận biết KITTI hay nuScenes dựa trên cấu trúc thư mục.\n\n    from starter.datasets import list_frames, load_frame, load_points\n    frames = list_frames("data/kitti_mini")              # [\'000001\', \'000004\', ...]\n    frames = list_frames("data/nuscenes_mini_subset")    # [\'scene-0103_000\', ...]\n    fr = load_frame("data/kitti_mini", frames[0])        # dict: points, calib, image, labels\n\n- KITTI: thư mục có training/velodyne/*.bin (data/synthetic, data/kitti_mini).\n- nuScenes: thư mục có v1.0-mini/sample.json (data/nuscenes_mini_subset).\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom starter import kitti_io, nuscenes_io\n\n\ndef dataset_type(data_root: str | Path) -> str:\n    if nuscenes_io.is_nuscenes_root(data_root):\n        return "nuscenes"\n    if (Path(data_root) / "training" / "velodyne").is_dir():\n        return "kitti"\n    raise FileNotFoundError(\n        f"\'{data_root}\' không giống KITTI (thiếu training/velodyne/) cũng không giống nuScenes "\n        f"(thiếu v1.0-mini/sample.json). Kiểm tra lại đường dẫn --data-root hoặc xem data/README.md.")\n\n\ndef list_frames(data_root: str | Path) -> list[str]:\n    if dataset_type(data_root) == "nuscenes":\n        return nuscenes_io.list_frames(data_root)\n    return kitti_io.list_frames(data_root)\n\n\ndef load_points(data_root: str | Path, frame_id: str) -> np.ndarray:\n    """Chỉ đọc point cloud (N, 4), nhanh hơn load_frame khi không cần ảnh/label."""\n    if dataset_type(data_root) == "nuscenes":\n        return nuscenes_io.load_lidar(nuscenes_io.lidar_path(data_root, frame_id))\n    return kitti_io.load_velodyne(kitti_io.frame_paths(data_root, frame_id)["velodyne"])\n\n\ndef load_frame(data_root: str | Path, frame_id: str, **kwargs) -> dict:\n    """kwargs chỉ dùng cho nuScenes: camera="CAM_FRONT", use_ego_motion=True."""\n    if dataset_type(data_root) == "nuscenes":\n        return nuscenes_io.load_frame(data_root, frame_id, **kwargs)\n    return kitti_io.load_frame(data_root, frame_id)\n', 'starter/kitti_io.py': '"""Đọc dữ liệu định dạng KITTI 3D object (velodyne, calib, image_2, label_2).\n\nQuy ước KITTI cần nhớ:\n- Velodyne frame: x forward, y left, z up (đơn vị mét).\n- Camera (rectified) frame: x right, y down, z forward.\n- Label 3D nằm trong rectified camera frame, `location` là BOTTOM center của box,\n  `dimensions` theo thứ tự (h, w, l), `rotation_y` quay quanh trục y của camera.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass KittiCalib:\n    P2: np.ndarray              # (3, 4) projection của camera trái màu (image_2), đã rectify\n    R0_rect: np.ndarray         # (3, 3) rectification\n    Tr_velo_to_cam: np.ndarray  # (3, 4) extrinsic: velodyne -> camera 0 (chưa rectify)\n\n    @property\n    def T_cam_velo(self) -> np.ndarray:\n        """Ma trận 4x4 đưa điểm từ velodyne frame sang rectified camera frame."""\n        R0 = np.eye(4)\n        R0[:3, :3] = self.R0_rect\n        Tr = np.eye(4)\n        Tr[:3, :] = self.Tr_velo_to_cam\n        return R0 @ Tr\n\n\n@dataclass\nclass KittiObject:\n    type: str\n    truncated: float\n    occluded: int\n    alpha: float\n    bbox: np.ndarray        # (4,) x1, y1, x2, y2 pixel\n    dimensions: np.ndarray  # (3,) h, w, l mét\n    location: np.ndarray    # (3,) x, y, z bottom center, rectified camera frame\n    rotation_y: float\n    score: float | None = None\n\n\ndef frame_paths(data_root: str | Path, frame_id: str, split: str = "training") -> dict[str, Path]:\n    root = Path(data_root) / split\n    return {\n        "velodyne": root / "velodyne" / f"{frame_id}.bin",\n        "calib": root / "calib" / f"{frame_id}.txt",\n        "image": root / "image_2" / f"{frame_id}.png",\n        "label": root / "label_2" / f"{frame_id}.txt",\n    }\n\n\ndef list_frames(data_root: str | Path, split: str = "training") -> list[str]:\n    return sorted(p.stem for p in (Path(data_root) / split / "velodyne").glob("*.bin"))\n\n\ndef load_velodyne(path: str | Path) -> np.ndarray:\n    """Trả về (N, 4) float32: x, y, z, reflectance. Không tự lọc NaN."""\n    raw = np.fromfile(path, dtype=np.float32)\n    if raw.size % 4 != 0:\n        raise ValueError(f"{path}: số float32 ({raw.size}) không chia hết cho 4 -> sai format?")\n    return raw.reshape(-1, 4)\n\n\ndef load_calib(path: str | Path) -> KittiCalib:\n    values: dict[str, np.ndarray] = {}\n    for line in Path(path).read_text().splitlines():\n        if ":" not in line:\n            continue\n        key, val = line.split(":", 1)\n        values[key.strip()] = np.array(val.split(), dtype=np.float64)\n\n    r0 = values.get("R0_rect", values.get("R_rect"))\n    tr = values.get("Tr_velo_to_cam", values.get("Tr_velo_cam"))\n    if r0 is None or tr is None or "P2" not in values:\n        raise KeyError(f"{path}: thiếu P2 / R0_rect / Tr_velo_to_cam")\n    return KittiCalib(\n        P2=values["P2"].reshape(3, 4),\n        R0_rect=r0.reshape(3, 3),\n        Tr_velo_to_cam=tr.reshape(3, 4),\n    )\n\n\ndef load_image(path: str | Path) -> np.ndarray:\n    """Ảnh BGR uint8 (H, W, 3)."""\n    img = cv2.imread(str(path), cv2.IMREAD_COLOR)\n    if img is None:\n        raise FileNotFoundError(path)\n    return img\n\n\ndef load_labels(path: str | Path) -> list[KittiObject]:\n    objects = []\n    path = Path(path)\n    if not path.exists():\n        return objects\n    for line in path.read_text().splitlines():\n        f = line.split()\n        if not f or f[0] == "DontCare":\n            continue\n        objects.append(KittiObject(\n            type=f[0],\n            truncated=float(f[1]),\n            occluded=int(float(f[2])),\n            alpha=float(f[3]),\n            bbox=np.array(f[4:8], dtype=np.float64),\n            dimensions=np.array(f[8:11], dtype=np.float64),\n            location=np.array(f[11:14], dtype=np.float64),\n            rotation_y=float(f[14]),\n            score=float(f[15]) if len(f) > 15 else None,\n        ))\n    return objects\n\n\ndef load_frame(data_root: str | Path, frame_id: str, split: str = "training") -> dict:\n    p = frame_paths(data_root, frame_id, split)\n    return {\n        "frame_id": frame_id,\n        "points": load_velodyne(p["velodyne"]),\n        "calib": load_calib(p["calib"]),\n        "image": load_image(p["image"]),\n        "labels": load_labels(p["label"]),\n    }\n', 'starter/nuscenes_io.py': '"""Đọc nuScenes (data/nuscenes_mini_subset, bản v1.0-mini đã lọc) và trả về cùng cấu trúc với KITTI.\n\nKhông cần cài nuscenes-devkit. Hàm `load_frame` trả về dict giống `kitti_io.load_frame`:\npoints, calib (KittiCalib), image, labels (KittiObject), nên code projection viết cho KITTI\nchạy được nguyên vẹn trên nuScenes.\n\nKhác biệt quan trọng so với KITTI, cần nhớ khi phân tích:\n- LiDAR frame của nuScenes: x sang PHẢI, y về PHÍA TRƯỚC, z lên trên (KITTI: x phía trước, y sang trái).\n  Vì vậy azimuth 0° trong data_health.py ở nuScenes là hướng bên phải xe, không phải phía trước.\n- Point cloud có 5 cột (x, y, z, intensity 0–255, ring index). Ở đây intensity được chia 255 để cùng thang 0–1 với KITTI.\n- LiDAR và camera chụp ở hai thời điểm khác nhau (lệch tới vài chục ms), nên chuỗi biến đổi phải đi qua\n  global frame bằng ego pose ở từng thời điểm:\n      cam <- ego(t_cam) <- global <- ego(t_lidar) <- lidar\n  Truyền `use_ego_motion=False` để bỏ qua bước này (giả định hai sensor chụp cùng lúc) và xem lỗi Time.\n- frame id có dạng "<scene>_<số thứ tự keyframe 3 chữ số>", ví dụ "scene-0103_000".\n"""\nfrom __future__ import annotations\n\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom starter.kitti_io import KittiCalib, KittiObject, load_image\n\nVERSION = "v1.0-mini"\nCATEGORY_MAP = {\n    "vehicle.car": "Car", "vehicle.truck": "Truck", "vehicle.trailer": "Trailer",\n    "vehicle.bus.bendy": "Bus", "vehicle.bus.rigid": "Bus", "vehicle.construction": "ConstructionVehicle",\n    "vehicle.bicycle": "Bicycle", "vehicle.motorcycle": "Motorcycle",\n    "human.pedestrian.adult": "Pedestrian", "human.pedestrian.child": "Pedestrian",\n    "human.pedestrian.construction_worker": "Pedestrian", "human.pedestrian.police_officer": "Pedestrian",\n    "movable_object.barrier": "Barrier", "movable_object.trafficcone": "TrafficCone",\n}\n# nuScenes visibility (1: 0–40%, 2: 40–60%, 3: 60–80%, 4: 80–100% nhìn thấy) -> KITTI occluded (0 = thấy rõ)\nVISIBILITY_TO_OCCLUDED = {"1": 2, "2": 2, "3": 1, "4": 0}\n\n\ndef is_nuscenes_root(data_root: str | Path) -> bool:\n    return (Path(data_root) / VERSION / "sample.json").exists()\n\n\ndef _quat_to_rot(q) -> np.ndarray:\n    w, x, y, z = q\n    return np.array([\n        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],\n        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],\n        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],\n    ])\n\n\ndef _pose(rec: dict) -> np.ndarray:\n    T = np.eye(4)\n    T[:3, :3] = _quat_to_rot(rec["rotation"])\n    T[:3, 3] = rec["translation"]\n    return T\n\n\nclass _Tables:\n    def __init__(self, root: Path):\n        def load(name):\n            return json.loads((root / VERSION / f"{name}.json").read_text(encoding="utf-8"))\n\n        self.root = root\n        idx = {name: {r["token"]: r for r in load(name)} for name in\n               ("scene", "sample", "sample_data", "calibrated_sensor", "sensor", "ego_pose",\n                "sample_annotation", "instance", "category", "visibility")}\n        self.__dict__.update(idx)\n        self.data_of_sample: dict[str, dict[str, dict]] = {}\n        for d in self.sample_data.values():\n            if d["is_key_frame"]:\n                channel = self.sensor[self.calibrated_sensor[d["calibrated_sensor_token"]]["sensor_token"]]["channel"]\n                self.data_of_sample.setdefault(d["sample_token"], {})[channel] = d\n        self.anns_of_sample: dict[str, list[dict]] = {}\n        for a in self.sample_annotation.values():\n            self.anns_of_sample.setdefault(a["sample_token"], []).append(a)\n        self.frames: dict[str, str] = {}\n        for scene in sorted(self.scene.values(), key=lambda s: s["name"]):\n            token, i = scene["first_sample_token"], 0\n            while token:\n                self.frames[f"{scene[\'name\']}_{i:03d}"] = token\n                token, i = self.sample[token]["next"], i + 1\n\n\n@lru_cache(maxsize=4)\ndef _tables(data_root: str) -> _Tables:\n    return _Tables(Path(data_root))\n\n\ndef list_frames(data_root: str | Path) -> list[str]:\n    return list(_tables(str(data_root)).frames)\n\n\ndef lidar_path(data_root: str | Path, frame_id: str) -> Path:\n    t = _tables(str(data_root))\n    return t.root / t.data_of_sample[t.frames[frame_id]]["LIDAR_TOP"]["filename"]\n\n\ndef load_lidar(path: str | Path) -> np.ndarray:\n    """(N, 4) float32 x, y, z, intensity (đã chia 255). Ring index bị bỏ để cùng format với KITTI."""\n    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)[:, :4].copy()\n    pts[:, 3] /= 255.0\n    return pts\n\n\ndef load_frame(data_root: str | Path, frame_id: str, camera: str = "CAM_FRONT",\n               use_ego_motion: bool = True) -> dict:\n    t = _tables(str(data_root))\n    if frame_id not in t.frames:\n        raise KeyError(f"Không có frame \'{frame_id}\'. Ví dụ frame hợp lệ: {list(t.frames)[:3]}")\n    sample_token = t.frames[frame_id]\n    lidar, cam = t.data_of_sample[sample_token]["LIDAR_TOP"], t.data_of_sample[sample_token][camera]\n    cs_lidar, cs_cam = t.calibrated_sensor[lidar["calibrated_sensor_token"]], t.calibrated_sensor[cam["calibrated_sensor_token"]]\n    ego_lidar = _pose(t.ego_pose[lidar["ego_pose_token"]])\n    ego_cam = _pose(t.ego_pose[cam["ego_pose_token"]]) if use_ego_motion else ego_lidar\n\n    T_cam_global = np.linalg.inv(_pose(cs_cam)) @ np.linalg.inv(ego_cam)\n    T_cam_lidar = T_cam_global @ ego_lidar @ _pose(cs_lidar)\n    K = np.array(cs_cam["camera_intrinsic"])\n    calib = KittiCalib(P2=np.hstack([K, np.zeros((3, 1))]), R0_rect=np.eye(3), Tr_velo_to_cam=T_cam_lidar[:3, :])\n\n    image = load_image(t.root / cam["filename"])\n    return {\n        "frame_id": frame_id,\n        "points": load_lidar(t.root / lidar["filename"]),\n        "calib": calib,\n        "image": image,\n        "labels": _labels(t, sample_token, T_cam_global, K, image.shape),\n        "timestamp_lidar_us": lidar["timestamp"],\n        "timestamp_camera_us": cam["timestamp"],\n    }\n\n\ndef _labels(t: _Tables, sample_token: str, T_cam_global: np.ndarray, K: np.ndarray,\n            image_shape) -> list[KittiObject]:\n    from starter.projection import box3d_corners_cam\n\n    h_img, w_img = image_shape[:2]\n    R_cg = T_cam_global[:3, :3]\n    objects = []\n    for a in t.anns_of_sample.get(sample_token, []):\n        category = t.category[t.instance[a["instance_token"]]["category_token"]]["name"]\n        w, l, h = a["size"]\n        center = T_cam_global @ np.r_[a["translation"], 1.0]\n        bottom = center[:3] + R_cg @ np.array([0.0, 0.0, -h / 2])\n        heading = R_cg @ _quat_to_rot(a["rotation"]) @ np.array([1.0, 0.0, 0.0])\n        ry = float(np.arctan2(-heading[2], heading[0]))\n        obj = KittiObject(type=CATEGORY_MAP.get(category, category.split(".")[-1]), truncated=0.0,\n                          occluded=VISIBILITY_TO_OCCLUDED.get(a["visibility_token"], 3),\n                          alpha=ry - float(np.arctan2(bottom[0], bottom[2])), bbox=np.zeros(4),\n                          dimensions=np.array([h, w, l]), location=bottom, rotation_y=ry)\n        corners = box3d_corners_cam(obj)\n        if (corners[:, 2] <= 0.1).any():\n            continue\n        uv = (K @ corners.T).T\n        uv = uv[:, :2] / uv[:, 2:3]\n        x1, y1 = uv.min(0)\n        x2, y2 = uv.max(0)\n        cx1, cy1, cx2, cy2 = max(x1, 0), max(y1, 0), min(x2, w_img - 1), min(y2, h_img - 1)\n        if cx1 >= cx2 or cy1 >= cy2:\n            continue\n        obj.bbox = np.array([cx1, cy1, cx2, cy2])\n        obj.truncated = float(1 - (cx2 - cx1) * (cy2 - cy1) / max((x2 - x1) * (y2 - y1), 1e-6))\n        objects.append(obj)\n    return objects\n', 'starter/perturb.py': '"""Các phép làm xấu point cloud cho topic C (degradation stress test).\n\nMọi hàm nhận/trả (N, 4) float32 [x, y, z, intensity] và không sửa input tại chỗ.\nLuôn truyền `seed` để kết quả tái lập được giữa các lần chạy.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef random_dropout(points: np.ndarray, keep_ratio: float, seed: int = 0) -> np.ndarray:\n    rng = np.random.default_rng(seed)\n    return points[rng.random(len(points)) < keep_ratio]\n\n\ndef range_dropout(points: np.ndarray, max_range_m: float) -> np.ndarray:\n    return points[np.linalg.norm(points[:, :2], axis=1) <= max_range_m]\n\n\ndef sector_dropout(points: np.ndarray, az_start_deg: float, az_end_deg: float) -> np.ndarray:\n    """Bỏ các điểm có azimuth trong [start, end] độ (0 = phía trước, dương = bên trái)."""\n    az = np.degrees(np.arctan2(points[:, 1], points[:, 0]))\n    return points[~((az >= az_start_deg) & (az <= az_end_deg))]\n\n\ndef beam_dropout(points: np.ndarray, keep_every: int = 2, n_beams: int = 64,\n                 fov_deg: tuple[float, float] = (-24.9, 2.0)) -> np.ndarray:\n    """Giả lập LiDAR ít beam hơn bằng cách gán beam theo góc elevation rồi giữ 1/keep_every."""\n    elev = np.degrees(np.arctan2(points[:, 2], np.linalg.norm(points[:, :2], axis=1)))\n    beam = np.clip(((elev - fov_deg[0]) / (fov_deg[1] - fov_deg[0]) * n_beams).astype(int), 0, n_beams - 1)\n    return points[beam % keep_every == 0]\n\n\ndef gaussian_noise(points: np.ndarray, sigma_xyz_m: float = 0.02, sigma_intensity: float = 0.0,\n                   seed: int = 0) -> np.ndarray:\n    rng = np.random.default_rng(seed)\n    out = points.copy()\n    out[:, :3] += rng.normal(0, sigma_xyz_m, size=(len(out), 3)).astype(out.dtype)\n    if sigma_intensity:\n        out[:, 3] = np.clip(out[:, 3] + rng.normal(0, sigma_intensity, len(out)), 0, 1)\n    return out\n\n\ndef motion_smear(points: np.ndarray, ego_speed_mps: float, sweep_time_s: float = 0.1) -> np.ndarray:\n    """Giả lập thiếu deskew: mỗi điểm bị dịch theo -x tỉ lệ với thời điểm quét của nó\n    (suy từ azimuth, LiDAR quay 360° trong sweep_time_s)."""\n    az = np.arctan2(points[:, 1], points[:, 0])\n    t = (az + np.pi) / (2 * np.pi) * sweep_time_s\n    out = points.copy()\n    out[:, 0] -= (ego_speed_mps * t).astype(out.dtype)\n    return out\n', 'starter/projection.py': '"""LiDAR -> camera projection, overlay và perturb calibration.\n\nHai hàm có `TODO(CP2)` là phần học viên phải tự cài đặt (bắt buộc cho topic A, C, F;\nkhuyến khích cho mọi topic vì đây là bài test calibration rẻ nhất).\n\nChạy thử sau khi cài đặt xong (cùng một code chạy được cho cả KITTI và nuScenes):\n    python -m starter.projection --data-root data/synthetic --frame 000000\n    python -m starter.projection --data-root data/synthetic --frame 000000 --yaw-deg 1.0\n    python -m starter.projection --data-root data/kitti_mini --frame 000011\n    python -m starter.projection --data-root data/nuscenes_mini_subset --frame scene-0103_010\n    python -m starter.projection --data-root data/nuscenes_mini_subset --frame scene-0103_010 --ignore-ego-motion\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom starter.datasets import dataset_type, load_frame\nfrom starter.kitti_io import KittiCalib, KittiObject\n\n\ndef velo_to_cam(points_xyz: np.ndarray, calib: KittiCalib) -> np.ndarray:\n    """Đưa điểm (N, 3) từ velodyne frame sang rectified camera frame (N, 3).\n\n    TODO(CP2):\n      1. Chuyển sang toạ độ đồng nhất (N, 4).\n      2. Nhân với calib.T_cam_velo (4x4). Chú ý chiều nhân và transpose.\n      3. Trả về 3 cột đầu.\n    Tự kiểm: một điểm velodyne (10, 0, 0) phải có z_cam ~ 10 (phía trước camera).\n    """\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\n    if points_xyz.ndim != 2 or points_xyz.shape[1] != 3:\n        raise ValueError("points_xyz phải có shape (N, 3)")\n    transform = calib.T_cam_velo\n    if transform.shape != (4, 4) or not np.isfinite(transform).all():\n        raise ValueError("Calibration T_cam_velo phải là ma trận hữu hạn (4, 4)")\n    homogeneous = np.column_stack((points_xyz, np.ones(len(points_xyz))))\n    # Điểm lỗi được giữ nguyên thứ tự để cam_to_image loại bằng mask.\n    with np.errstate(invalid="ignore", over="ignore"):\n        return (homogeneous @ transform.T)[:, :3]\n\n\ndef cam_to_image(points_cam: np.ndarray, P2: np.ndarray, image_shape: tuple[int, ...],\n                 min_depth: float = 0.1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Chiếu điểm camera frame (N, 3) lên ảnh bằng P2 (3x4).\n\n    Trả về:\n      uv    (M, 2) toạ độ pixel của các điểm hợp lệ\n      depth (M,)   z_cam của các điểm hợp lệ\n      mask  (N,)   bool, True nếu điểm hợp lệ\n\n    Điểm hợp lệ = depth > min_depth VÀ nằm trong ảnh (0 <= u < W, 0 <= v < H).\n\n    TODO(CP2):\n      1. Lọc điểm không hợp lệ (NaN/Inf): dữ liệu thật không bao giờ sạch.\n      2. Toạ độ đồng nhất, nhân P2 -> (N, 3) = [s*u, s*v, s].\n      3. Chia cho s để có (u, v). Chỉ chia với điểm có depth > min_depth.\n      4. Lọc theo kích thước ảnh image_shape[:2] = (H, W).\n    """\n    points_cam = np.asarray(points_cam, dtype=np.float64)\n    P2 = np.asarray(P2, dtype=np.float64)\n    if points_cam.ndim != 2 or points_cam.shape[1] != 3 or P2.shape != (3, 4):\n        raise ValueError("Cần points_cam (N, 3) và P2 (3, 4)")\n    if not np.isfinite(P2).all():\n        raise ValueError("Calibration P2 chứa NaN/Inf")\n    if not np.isfinite(min_depth) or min_depth < 0:\n        raise ValueError("min_depth phải hữu hạn và không âm")\n    height, width = image_shape[:2]\n    if height <= 0 or width <= 0:\n        raise ValueError("Kích thước ảnh phải dương")\n    mask = np.zeros(len(points_cam), dtype=bool)\n    candidate = np.flatnonzero(np.isfinite(points_cam).all(axis=1)\n                               & (points_cam[:, 2] > min_depth))\n    homogeneous = np.column_stack((points_cam[candidate], np.ones(len(candidate))))\n    projected = homogeneous @ P2.T\n    # s là mẫu số homogeneous, có thể khác z_cam khi P2 có translation.\n    safe = np.isfinite(projected).all(axis=1) & (projected[:, 2] > 1e-12)\n    candidate = candidate[safe]\n    projected = projected[safe]\n    uv = projected[:, :2] / projected[:, 2:3]\n    inside = (np.isfinite(uv).all(axis=1) & (uv[:, 0] >= 0) & (uv[:, 0] < width)\n              & (uv[:, 1] >= 0) & (uv[:, 1] < height))\n    mask[candidate[inside]] = True\n    return uv[inside], points_cam[candidate[inside], 2], mask\n\n\ndef project_velo_to_image(points: np.ndarray, calib: KittiCalib, image_shape: tuple[int, ...]):\n    """points (N, >=3) velodyne -> (uv, depth, mask) như `cam_to_image`."""\n    return cam_to_image(velo_to_cam(points[:, :3], calib), calib.P2, image_shape)\n\n\ndef overlay_points(image: np.ndarray, uv: np.ndarray, depth: np.ndarray,\n                   max_depth: float = 50.0, radius: int = 2) -> np.ndarray:\n    """Vẽ điểm lên ảnh, màu theo depth (gần = đỏ, xa = xanh)."""\n    out = image.copy()\n    d = np.clip(depth / max_depth, 0, 1)\n    colors = cv2.applyColorMap((255 * (1 - d)).astype(np.uint8).reshape(-1, 1), cv2.COLORMAP_JET)\n    for (u, v), c in zip(uv.astype(int), colors[:, 0]):\n        cv2.circle(out, (int(u), int(v)), radius, tuple(int(x) for x in c), -1)\n    return out\n\n\ndef _rot(roll: float, pitch: float, yaw: float) -> np.ndarray:\n    cr, sr, cp, sp, cy, sy = np.cos(roll), np.sin(roll), np.cos(pitch), np.sin(pitch), np.cos(yaw), np.sin(yaw)\n    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])\n    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])\n    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])\n    return Rz @ Ry @ Rx\n\n\ndef perturb_extrinsic(calib: KittiCalib, roll_deg: float = 0.0, pitch_deg: float = 0.0,\n                      yaw_deg: float = 0.0, t_xyz_m: tuple[float, float, float] = (0, 0, 0)) -> KittiCalib:\n    """Giả lập calibration drift: xoay/dịch LiDAR trong chính velodyne frame\n    (yaw quanh z-up, pitch quanh y-left, roll quanh x-forward) rồi ghép vào Tr_velo_to_cam."""\n    D = np.eye(4)\n    D[:3, :3] = _rot(*np.deg2rad([roll_deg, pitch_deg, yaw_deg]))\n    D[:3, 3] = t_xyz_m\n    Tr = np.eye(4)\n    Tr[:3, :] = calib.Tr_velo_to_cam\n    out = copy.deepcopy(calib)\n    out.Tr_velo_to_cam = (Tr @ D)[:3, :]\n    return out\n\n\ndef box3d_corners_cam(obj: KittiObject) -> np.ndarray:\n    """8 góc (8, 3) của box KITTI trong rectified camera frame.\n    Thứ tự: 4 góc đáy (y=0) rồi 4 góc nóc (y=-h)."""\n    h, w, l = obj.dimensions\n    x = np.array([l, l, -l, -l, l, l, -l, -l]) / 2\n    y = np.array([0, 0, 0, 0, -h, -h, -h, -h])\n    z = np.array([w, -w, -w, w, w, -w, -w, w]) / 2\n    c, s = np.cos(obj.rotation_y), np.sin(obj.rotation_y)\n    R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])\n    return (R @ np.vstack([x, y, z])).T + obj.location\n\n\ndef draw_box2d(image: np.ndarray, bbox, color=(0, 255, 0), label: str | None = None) -> np.ndarray:\n    out = image.copy()\n    x1, y1, x2, y2 = (int(round(v)) for v in bbox)\n    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)\n    if label:\n        cv2.putText(out, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Chiếu điểm LiDAR lên ảnh camera, tô màu theo độ sâu, vẽ 2D box của label")\n    ap.add_argument("--data-root", default="data/synthetic",\n                    help="thư mục KITTI (data/synthetic, data/kitti_mini) hoặc nuScenes (data/nuscenes_mini_subset)")\n    ap.add_argument("--frame", default="000000", help="KITTI: 000011. nuScenes: scene-0103_010")\n    ap.add_argument("--out-dir", default="results/figures", help="nơi lưu ảnh overlay")\n    ap.add_argument("--roll-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục x (độ)")\n    ap.add_argument("--pitch-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục y (độ)")\n    ap.add_argument("--yaw-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục z hướng lên (độ)")\n    ap.add_argument("--tx", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục x của LiDAR (mét)")\n    ap.add_argument("--ty", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục y của LiDAR (mét)")\n    ap.add_argument("--tz", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục z của LiDAR (mét)")\n    ap.add_argument("--ignore-ego-motion", action="store_true",\n                    help="chỉ cho nuScenes: bỏ bù chuyển động xe giữa thời điểm chụp LiDAR và camera")\n    args = ap.parse_args()\n\n    kwargs = {"use_ego_motion": not args.ignore_ego_motion} if dataset_type(args.data_root) == "nuscenes" else {}\n    fr = load_frame(args.data_root, args.frame, **kwargs)\n    calib = perturb_extrinsic(fr["calib"], args.roll_deg, args.pitch_deg, args.yaw_deg,\n                              (args.tx, args.ty, args.tz))\n    uv, depth, mask = project_velo_to_image(fr["points"], calib, fr["image"].shape)\n    vis = overlay_points(fr["image"], uv, depth)\n    for obj in fr["labels"]:\n        vis = draw_box2d(vis, obj.bbox, label=obj.type)\n\n    tag = f"r{args.roll_deg}_p{args.pitch_deg}_y{args.yaw_deg}_t{args.tx}_{args.ty}_{args.tz}"\n    if args.ignore_ego_motion:\n        tag += "_noego"\n    out = Path(args.out_dir) / f"overlay_{args.frame}_{tag}.png"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    cv2.imwrite(str(out), vis)\n    print(f"points={len(mask)} inside_image={int(mask.sum())} ({mask.mean():.1%}) -> {out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/__init__.py': '"""Student implementation for Topic A: LiDAR-camera calibration QA."""\n', 'src/calibration_benchmark.py': '"""CPU topic A benchmark. Run from repo root; never reuses an output directory.\n\nScore uses only candidate projections and image edges, never a reference calibration\nor reference UV. Fixed random point IDs are selected from finite input XYZ before\nprojection. Only candidate-visible samples enter the median; coverage is reported.\nThis heuristic can miss drift on textureless/repetitive scenes or changing FOV.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nfrom starter.datasets import list_frames, load_frame\nfrom starter.projection import velo_to_cam, cam_to_image, perturb_extrinsic\n\nANGLES = [-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3]\nTRANSLATIONS = [-0.1, -0.05, -0.02, 0, 0.02, 0.05, 0.1]\nROOTS = [\'data/kitti_mini\', \'data/nuscenes_mini_subset\', \'data/synthetic\']\nMIN_SCORE_VISIBLE = 30  # Fixed before experiments; never selected using test data.\nNATIVE_AXES = {\n    \'kitti_mini\': \'LiDAR native: x forward, y left, z up\',\n    \'synthetic\': \'LiDAR native: x forward, y left, z up (KITTI convention)\',\n    \'nuscenes_mini_subset\': \'LiDAR native: x right, y forward, z up\',\n}\nCAVEATS = [\n    \'nuScenes 2D bbox is generated from 3D labels, not independent 2D ground truth.\',\n    \'nuScenes KITTI-style upright box conversion is approximate (full orientation is lost).\',\n    \'Ego motion compensation is enabled; dynamic object motion between sensor timestamps is not compensated.\',\n    \'Membership is fixed per baseline object; overlapping 3D boxes may share points.\',\n    \'Frame split is within-seen-scenes only, deterministic and stratified by dataset/scene; adjacent frames are correlated. No unseen-scene generalization claim.\',\n    \'Synthetic calibration perturbations measure sensitivity, not validated real-world drift detection.\',\n    \'Canny score median uses candidate-visible fixed samples; changing coverage can bias it. Lower is better.\',\n    \'Canny heuristic matches projected points to image edges, not depth-edge to image-edge alignment; not a validated alignment metric.\',\n    \'Fewer than 30 candidate-visible fixed samples: coverage_insufficient, unscorable; fixed rule, no test tuning.\',\n    \'Perturbation axes are LiDAR native. Yaw is about z-up in both datasets; roll/pitch and x/y translations do not represent the same physical direction across KITTI and nuScenes.\',\n    \'Zero is an unperturbed control, not proof that dataset calibration/labels are perfect.\',\n]\n\n\ndef configurations():\n    out = [dict(config=\'zero\', factor=\'zero\', value=0.0, unit=\'none\')]\n    for factor in [\'yaw\', \'pitch\', \'roll\', \'x\', \'y\', \'z\']:\n        for value in ANGLES if factor in [\'yaw\', \'pitch\', \'roll\'] else TRANSLATIONS:\n            if value != 0:\n                out.append(dict(config=f\'{factor}_{value:+g}\', factor=factor,\n                                value=float(value), unit=\'deg\' if factor in [\'yaw\', \'pitch\', \'roll\'] else \'m\'))\n    return out\n\n\ndef drift(calib, cfg):\n    if cfg[\'factor\'] == \'zero\':\n        return calib\n    factor, value = cfg[\'factor\'], cfg[\'value\']\n    if factor in [\'yaw\', \'pitch\', \'roll\']:\n        return perturb_extrinsic(calib, **{factor + \'_deg\': value})\n    t = [0.0, 0.0, 0.0]\n    t[[\'x\', \'y\', \'z\'].index(factor)] = value\n    return perturb_extrinsic(calib, t_xyz_m=tuple(t))\n\n\ndef project(points, calib, shape):\n    uv, depth, mask = cam_to_image(velo_to_cam(points[:, :3], calib), calib.P2, shape)\n    mask = np.asarray(mask, dtype=bool)\n    if mask.shape != (len(points),) or len(uv) != mask.sum():\n        raise ValueError(\'Projection must preserve input order and return an N-length boolean mask.\')\n    if not np.isfinite(uv).all() or not np.isfinite(depth).all():\n        raise ValueError(\'Projection returned nonfinite valid UV/depth.\')\n    dense = np.full((len(points), 2), np.nan)\n    dense[mask] = uv\n    return dense, mask, np.asarray(depth)\n\n\ndef membership(points_cam, obj):\n    """Inverse KITTI yaw, bottom center: x ±l/2, y [-h,0], z ±w/2."""\n    h, w, length = obj.dimensions\n    c, s = np.cos(obj.rotation_y), np.sin(obj.rotation_y)\n    rotation = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])\n    local = (points_cam - obj.location) @ rotation\n    return (np.isfinite(local).all(1) & (np.abs(local[:, 0]) <= length / 2)\n            & (local[:, 1] >= -h) & (local[:, 1] <= 0)\n            & (np.abs(local[:, 2]) <= w / 2))\n\n\ndef edge_map(image):\n    edges = cv2.Canny(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), 100, 200)\n    # No edges: normalized distance 1, not a misleading perfect score.\n    if not edges.any():\n        return np.full(edges.shape, np.hypot(*edges.shape), np.float32)\n    return cv2.distanceTransform((edges == 0).astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE)\n\n\ndef edge_score(dense, valid, ids, distances):\n    selected = ids[valid[ids]]\n    if len(selected) < MIN_SCORE_VISIBLE:\n        return float(\'nan\'), len(selected)\n    pixels = np.floor(dense[selected]).astype(int)\n    values = distances[pixels[:, 1], pixels[:, 0]] / np.hypot(*distances.shape)\n    return float(np.median(values)), len(selected)\n\n\ndef split_frames(dataset, frames, seed):\n    result = {}\n    groups = {}\n    for frame in frames:\n        scene = frame.rsplit(\'_\', 1)[0] if dataset == \'nuscenes_mini_subset\' else dataset\n        groups.setdefault(scene, []).append(frame)\n    for scene, group in sorted(groups.items()):\n        digest = hashlib.sha256(f\'{seed}:{dataset}:{scene}\'.encode()).digest()\n        rng = np.random.default_rng(int.from_bytes(digest[:8], \'little\'))\n        order = rng.permutation(sorted(group))\n        n = max(1, min(len(group) - 1, len(group) // 2)) if len(group) > 1 else 1\n        result.update({str(frame): \'calibration\' if i < n else \'test\' for i, frame in enumerate(order)})\n    return result\n\n\ndef environment():\n    cpu = platform.processor()\n    cpuinfo = Path(\'/proc/cpuinfo\')\n    if cpuinfo.exists():\n        cpu = next((line.split(\':\', 1)[1].strip() for line in cpuinfo.read_text().splitlines()\n                    if line.startswith(\'model name\')), cpu)\n    return dict(python=sys.version, platform=platform.platform(), cpu=cpu,\n                logical_cpus=os.cpu_count(), numpy=np.__version__, opencv=cv2.__version__,\n                pandas=pd.__version__, opencv_threads=cv2.getNumThreads(),\n                code_sha256=code_hashes(),\n                projection_sha256=file_sha256(Path(\'starter/projection.py\')),\n                thread_environment={key: os.environ.get(key) for key in\n                                    [\'OPENBLAS_NUM_THREADS\', \'OMP_NUM_THREADS\', \'MKL_NUM_THREADS\']})\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef code_hashes():\n    paths = [\'src/calibration_benchmark.py\', \'starter/projection.py\',\n             \'starter/datasets.py\', \'starter/kitti_io.py\', \'starter/nuscenes_io.py\']\n    return {path: file_sha256(path) for path in paths}\n\n\ndef data_manifest(datasets):\n    return [dict(dataset=root.name, path=str(path), relative_path=str(path.relative_to(root)),\n                 bytes=path.stat().st_size, sha256=file_sha256(path))\n            for root, _ in datasets for path in sorted(root.rglob(\'*\')) if path.is_file()]\n\n\ndef write_json(path, value):\n    # Exclusive creation also protects individual artifacts.\n    with path.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n\n\ndef json_records(table):\n    return json.loads(table.to_json(orient=\'records\'))\n\n\ndef csv_payloads(table, filename, max_bytes=20_000_000, shard_rows=20000):\n    """Preserve the unsuffixed API below the limit; UTF-8 bytes strictly <20 MB."""\n    payload = table.to_csv(index=False).encode(\'utf-8\')\n    if len(payload) < max_bytes:\n        yield filename, len(table), payload\n        return\n    del payload\n    parts = []\n    def divide(chunk):\n        data = chunk.to_csv(index=False).encode(\'utf-8\')\n        if len(data) < max_bytes:\n            parts.append((len(chunk), data))\n        elif len(chunk) > 1:\n            mid = len(chunk) // 2\n            divide(chunk.iloc[:mid])\n            divide(chunk.iloc[mid:])\n        else:\n            raise ValueError(\'Single CSV row/header exceeds byte limit; cannot shard safely.\')\n    for start in range(0, len(table), shard_rows):\n        divide(table.iloc[start:start + shard_rows])\n    stem = Path(filename).stem\n    for index, (rows, data) in enumerate(parts):\n        yield f\'{stem}_{index:03d}.csv\', rows, data\n\n\ndef write_csv(table, path):\n    artifacts = []\n    for filename, rows, payload in csv_payloads(table, path.name):\n        with (path.parent / filename).open(\'xb\') as stream:\n            stream.write(payload)\n        artifacts.append(dict(file=filename, rows=rows, bytes=len(payload),\n                              sha256=hashlib.sha256(payload).hexdigest()))\n    return artifacts\n\n\ndef summary(frames, objects):\n    rows = []\n    for (dataset, config), group in frames.groupby([\'dataset\', \'config\'], sort=False):\n        ob = objects[(objects.dataset == dataset) & (objects.config == config)] if len(objects) else objects\n        eligible = ob[ob.eligible] if len(ob) else ob\n        n, common = group.n_points.sum(), group.common_valid.sum()\n        den = eligible.baseline_visible_assigned.sum() if len(eligible) else 0\n        rows.append(dict(dataset=dataset, config=config, factor=group.factor.iloc[0],\n                         value=group.value.iloc[0], unit=group.unit.iloc[0], frames=len(group),\n                         fov_micro=group.inside_count.sum() / n if n else np.nan,\n                         fov_macro=group.inside_fov.mean(),\n                         displacement_px_micro=group.displacement_sum_px.sum() / common if common else np.nan,\n                         displacement_px_macro=group.displacement_mean_px.mean(),\n                         coverage_micro=common / group.baseline_inside_count.sum() if group.baseline_inside_count.sum() else np.nan,\n                         coverage_macro=group.common_coverage.mean(),\n                         association_micro=eligible.bbox_hits.sum() / den if den else np.nan,\n                         association_macro=eligible.association_ratio.mean() if len(eligible) else np.nan,\n                         eligible_objects=len(eligible), edge_score_macro=group.edge_score.mean(),\n                         score_coverage_micro=group.score_visible_count.sum() / group.score_sample_count.sum()\n                         if group.score_sample_count.sum() else np.nan,\n                         score_coverage_macro=group.score_coverage.mean()))\n    return pd.DataFrame(rows)\n\n\ndef detection_rates(group, threshold):\n    """Unscorable test rows count as misses; conditional recall is explicitly separate."""\n    available = group[group.score_available]\n    threshold_available = threshold is not None\n    rate = float(group.detected.mean()) if threshold_available and len(group) else None\n    conditional = float(available.detected.mean()) if threshold_available and len(available) else None\n    return dict(threshold=threshold, threshold_available=threshold_available,\n                unavailable_reason=None if threshold_available else \'No scorable zero calibration frames (minimum visible samples = 30).\',\n                n=len(group), n_score_available=len(available),\n                unscorable_rate=float((~group.score_available).mean()) if len(group) else None,\n                sensitivity_recall=rate, sensitivity_recall_conditional=conditional,\n                missed_or_unscorable=int((~group.detected).sum()) if threshold_available else None)\n\n\ndef detection(frames, out):\n    """95th percentile train zero only; test is never used to choose threshold."""\n    evaluations, roc_rows, thresholds = [], [], {}\n    for dataset, group in frames.groupby(\'dataset\'):\n        train = group[(group.split == \'calibration\') & (group.config == \'zero\')].edge_score.dropna()\n        threshold = float(np.quantile(train, .95)) if len(train) else None\n        thresholds[dataset] = threshold\n        test = group[group.split == \'test\'].copy()\n        test[\'threshold\'] = threshold\n        test[\'threshold_available\'] = threshold is not None\n        test[\'score_available\'] = test.edge_score.notna()\n        test[\'detected\'] = test.edge_score > threshold if threshold is not None else False\n        test[\'is_perturbed\'] = test.config != \'zero\'\n        evaluations.append(test)\n        finite = test[test.score_available]\n        negative, positive = finite[~finite.is_perturbed], finite[finite.is_perturbed]\n        candidates = np.r_[-np.inf, np.unique(finite.edge_score), np.inf]\n        for cut in candidates:\n            roc_rows.append(dict(dataset=dataset, threshold=cut,\n                                 fpr=float((negative.edge_score > cut).mean()) if len(negative) else np.nan,\n                                 recall=float((positive.edge_score > cut).mean()) if len(positive) else np.nan))\n    evaluation = pd.concat(evaluations, ignore_index=True)\n    write_csv(evaluation, out / \'score_test.csv\')\n    roc = pd.DataFrame(roc_rows)\n    write_csv(roc, out / \'score_roc.csv\')\n    reports = []\n    for (dataset, factor, value), group in evaluation.groupby([\'dataset\', \'factor\', \'value\']):\n        available = group[group.score_available]\n        rates = detection_rates(group, thresholds[dataset])\n        rates[\'false_positive_rate\'] = rates[\'sensitivity_recall\'] if factor == \'zero\' else None\n        rates[\'false_positive_rate_conditional\'] = rates[\'sensitivity_recall_conditional\'] if factor == \'zero\' else None\n        if factor == \'zero\':\n            rates[\'sensitivity_recall\'] = rates[\'sensitivity_recall_conditional\'] = None\n        reports.append(dict(dataset=dataset, factor=factor, value=float(value), **rates))\n    overall = [dict(dataset=dataset, **detection_rates(group[group.is_perturbed], thresholds[dataset]))\n               for dataset, group in evaluation.groupby(\'dataset\')]\n    write_json(out / \'score_evaluation.json\', dict(threshold_rule=\'baseline calibration quantile 0.95; strict score > threshold\',\n               thresholds=thresholds, results=reports, overall_test_drift=overall, caveats=CAVEATS,\n               min_score_visible=MIN_SCORE_VISIBLE,\n               interpretation=\'Report all rates as observed; no test tuning. Recall is synthetic sensitivity only.\'))\n    return evaluation, roc, thresholds\n\n\ndef plots(table, evaluation, roc, thresholds, out):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    for dataset, group in table.groupby(\'dataset\'):\n        metrics = [(\'fov_micro\', \'Inside FOV / all input points\'),\n                   (\'displacement_px_micro\', \'Common-valid displacement (px)\'),\n                   (\'association_micro\', \'Correct bbox / baseline-visible assigned\'),\n                   (\'coverage_micro\', \'Common valid / baseline valid\'),\n                   (\'edge_score_macro\', \'Median edge distance / image diagonal\'),\n                   (\'score_coverage_micro\', \'Visible score samples / fixed samples\')]\n        for kind, factors, unit in [(\'angles\', [\'yaw\', \'pitch\', \'roll\'], \'degrees\'),\n                                    (\'translation\', [\'x\', \'y\', \'z\'], \'meters\')]:\n            fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n            for ax, (metric, label) in zip(axes.flat, metrics):\n                baseline = group[group.config == \'zero\']\n                for factor in factors:\n                    curve = pd.concat([group[group.factor == factor], baseline]).sort_values(\'value\')\n                    ax.plot(curve.value, curve[metric], \'.-\', label=factor)\n                ax.set(title=label, xlabel=f\'Independent perturbation ({unit})\')\n                ax.legend(fontsize=8)\n                ax.grid(alpha=.2)\n            fig.suptitle(dataset + f\' — {kind}; micro metrics, edge score frame macro\')\n            fig.text(.5, .01, NATIVE_AXES.get(dataset, \'LiDAR native axes: verify custom dataset convention\')\n                     + \' | roll=x, pitch=y, yaw=z; x/y and roll/pitch are not physically equivalent across datasets\',\n                     ha=\'center\', fontsize=8)\n            fig.tight_layout(rect=(0, .035, 1, .96))\n            fig.savefig(out / f\'sweep_{dataset}_{kind}.png\', dpi=140)\n            plt.close(fig)\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    for dataset, group in roc.groupby(\'dataset\'):\n        axes[0].plot(group.fpr, group.recall, label=dataset)\n        test = evaluation[(evaluation.dataset == dataset) & evaluation.score_available]\n        for perturbed, symbol in [(False, \'o\'), (True, \'x\')]:\n            data = test[test.is_perturbed == perturbed]\n            axes[1].scatter(np.full(len(data), list(thresholds).index(dataset) + .15 * perturbed),\n                            data.edge_score, s=8, marker=symbol,\n                            label=f\'{dataset}: \' + (\'perturbed\' if perturbed else \'zero\'))\n        if thresholds[dataset] is not None:\n            i = list(thresholds).index(dataset)\n            axes[1].plot([i - .2, i + .35], [thresholds[dataset]] * 2, \'k-\')\n    axes[0].plot([0, 1], [0, 1], \'k--\', alpha=.3)\n    axes[0].set(xlabel=\'Test zero false-positive rate (scorable only)\', ylabel=\'Conditional recall (scorable only)\', title=\'ROC: descriptive; see JSON for all-test recall\')\n    axes[1].set(xticks=range(len(thresholds)), xticklabels=list(thresholds), ylabel=\'Normalized edge distance\', title=\'Calibration-only thresholds (black)\')\n    for ax in axes:\n        ax.legend(fontsize=6)\n        ax.grid(alpha=.2)\n    fig.tight_layout()\n    fig.savefig(out / \'score_roc_threshold.png\', dpi=140)\n    plt.close(fig)\n\n\ndef overlay(frame, cfg, title, max_points=6000, selected_object_id=None):\n    image = frame[\'image\'].copy()\n    dense, valid, depth = project(frame[\'points\'], drift(frame[\'calib\'], cfg), image.shape)\n    ids = np.flatnonzero(valid)\n    step = max(1, int(np.ceil(len(ids) / max_points)))\n    ids = ids[::step]\n    # Rendering is capped at 6000 circles per panel; full clouds are used for metrics.\n    if len(ids):\n        pixels = np.floor(dense[ids]).astype(int)\n        color = cv2.applyColorMap((255 * (1 - np.clip(depth[::step] / 70, 0, 1))).astype(np.uint8).reshape(-1, 1), cv2.COLORMAP_JET)[:, 0]\n        for (u, v), bgr in zip(pixels, color):\n            cv2.circle(image, (int(u), int(v)), 2, tuple(int(c) for c in bgr), -1)\n    for object_id, obj in enumerate(frame[\'labels\']):\n        x1, y1, x2, y2 = np.round(obj.bbox).astype(int)\n        distance = float(np.linalg.norm(obj.location))\n        selected = object_id == selected_object_id\n        box_color = (0, 255, 255) if selected else (0, 255, 0)\n        cv2.rectangle(image, (x1, y1), (x2, y2), box_color, 3 if selected else 1)\n        prefix = \'SELECTED \' if selected else \'\'\n        cv2.putText(image, f\'{prefix}#{object_id} {obj.type} {distance:.2f}m\',\n                    (max(0, x1), max(45, y1)), cv2.FONT_HERSHEY_SIMPLEX,\n                    .5 if selected else .42, box_color, 2 if selected else 1)\n    cv2.rectangle(image, (0, 0), (image.shape[1], 30), (0, 0, 0), -1)\n    cv2.putText(image, title, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, .5, (255, 255, 255), 1)\n    return image\n\n\ndef save_image(path, image):\n    if path.exists():\n        raise FileExistsError(path)\n    if not cv2.imwrite(str(path), image):\n        raise OSError(f\'Could not write {path}\')\n\n\ndef visual_artifacts(catalog, frames, evaluation, configs, out):\n    zero = configs[0]\n    manifest, used = [], set()\n    kitti = [item for item in catalog if item[\'dataset\'] == \'kitti_mini\' and item[\'object_distances_m\']]\n    for band, target in [(\'near\', 5), (\'mid\', 25), (\'far\', 60)]:\n        candidates = [item for item in kitti if item[\'frame\'] not in used]\n        if not candidates:\n            raise RuntimeError(\'Need three distinct labeled KITTI frames for near/mid/far overlays.\')\n        _, _, object_id, item = min(\n            (abs(distance - target), item[\'frame\'], object_id, item)\n            for item in candidates for object_id, distance in enumerate(item[\'object_distances_m\']))\n        actual_distance = item[\'object_distances_m\'][object_id]\n        used.add(item[\'frame\'])\n        frame = load_frame(item[\'root\'], item[\'frame\'])\n        name = f\'overlay_kitti_{band}_{item["frame"]}.png\'\n        save_image(out / name, overlay(frame, zero,\n                   f\'KITTI {band} | {item["frame"]} | selected #{object_id}: {actual_distance:.2f}m\',\n                   selected_object_id=object_id))\n        manifest.append(dict(file=name, selection_target_m=target, selected_object_id=object_id,\n                             selected_object_type=frame[\'labels\'][object_id].type,\n                             selected_actual_distance_m=actual_distance,\n                             distance_convention=\'Euclidean norm of 3D bottom-center in baseline rectified camera frame\',\n                             selection_rule=\'Closest object to target among frames not selected previously; no fixed distance bin claimed.\',\n                             **item))\n    for scene, light in [(\'scene-0103\', \'day\'), (\'scene-1094\', \'night\')]:\n        items = [item for item in catalog if item[\'dataset\'] == \'nuscenes_mini_subset\' and item[\'frame\'].startswith(scene)]\n        if not items:\n            raise RuntimeError(f\'Missing required nuScenes {light} scene {scene}\')\n        item = items[len(items) // 2]\n        name = f\'overlay_nuscenes_{light}_{item["frame"]}.png\'\n        save_image(out / name, overlay(load_frame(item[\'root\'], item[\'frame\']), zero,\n                                       f\'nuScenes {light} | {item["frame"]} | approx labels\'))\n        manifest.append(dict(file=name, **item))\n    write_json(out / \'overlay_manifest.json\', manifest)\n    # Prefer strongest missed test drift; otherwise strongest displacement with FOV delta <=1 pp.\n    candidate = evaluation[(evaluation.config != \'zero\') & (~evaluation.detected)\n                           & (evaluation.dataset != \'synthetic\') & (evaluation.displacement_mean_px > 0)].copy()\n    reason = \'Test score misses synthetic drift (or score unavailable); largest common-valid mean displacement.\'\n    if candidate.empty:\n        candidate = frames[(frames.config != \'zero\') & (frames.dataset != \'synthetic\')\n                           & (frames.fov_change.abs() <= .01)].copy()\n        reason = \'Inside FOV changes at most 1 percentage point; largest common-valid mean displacement.\'\n    if candidate.empty:\n        raise RuntimeError(\'No qualifying objective failure case; artifacts incomplete, do not invent a failure.\')\n    row = candidate.sort_values([\'displacement_mean_px\', \'dataset\', \'frame\', \'config\'], ascending=[False, True, True, True]).iloc[0]\n    item = next(x for x in catalog if x[\'dataset\'] == row.dataset and x[\'frame\'] == row.frame)\n    frame = load_frame(item[\'root\'], item[\'frame\'])\n    cfg = next(c for c in configs if c[\'config\'] == row.config)\n    left = overlay(frame, zero, f\'Baseline | {row.frame}\')\n    right = overlay(frame, cfg, f\'{row.config} | displacement={row.displacement_mean_px:.2f}px\')\n    name = f\'fail_01_{row.dataset}_{row.frame}_{row.config}.png\'\n    save_image(out / name, np.hstack([left, right]))\n    write_json(out / \'fail_01_facts.json\', dict(file=name, selection_rule=reason,\n               metrics=json_records(pd.DataFrame([row]))[0], source=item, caveats=CAVEATS))\n\n\ndef latency(frame, dataset, repeats, out_rows):\n    points, calib, shape = frame[\'points\'], frame[\'calib\'], frame[\'image\'].shape\n    for _ in range(5):\n        project(points, calib, shape)\n    samples = []\n    for i in range(repeats):\n        start = time.perf_counter_ns()\n        # Time only starter projection, not IO, drift construction, metrics or dense UV allocation.\n        cam_to_image(velo_to_cam(points[:, :3], calib), calib.P2, shape)\n        elapsed = (time.perf_counter_ns() - start) / 1e6\n        samples.append(elapsed)\n        out_rows.append(dict(dataset=dataset, frame=frame[\'frame_id\'], repeat=i, milliseconds=elapsed,\n                             n_points=len(points)))\n    return dict(dataset=dataset, frame=frame[\'frame_id\'], repeats=repeats, warmup=5,\n                p50_ms=float(np.percentile(samples, 50)), p95_ms=float(np.percentile(samples, 95)),\n                scope=\'velo_to_cam + cam_to_image; preloaded same input; CPU\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\'--out-dir\', required=True, type=Path, help=\'New artifact directory, e.g. results/topic_a_20261007_01. Existing directories are refused.\')\n    parser.add_argument(\'--data-roots\', nargs=\'+\', default=ROOTS, help=\'Default: all 20 KITTI, 80 nuScenes and 5 synthetic frames. No frame truncation.\')\n    parser.add_argument(\'--seed\', type=int, default=2026)\n    parser.add_argument(\'--score-samples\', type=int, default=4096, help=\'Fixed finite XYZ point IDs/frame, before any projection.\')\n    parser.add_argument(\'--latency-repeats\', type=int, default=30, help=\'At least 20, after five warmups.\')\n    args = parser.parse_args()\n    if args.score_samples < 1 or args.latency_repeats < 20:\n        parser.error(\'score-samples >=1 and latency-repeats >=20 required\')\n    if args.out_dir.exists():\n        parser.error(\'Output directory already exists; choose a fresh runid. Nothing overwritten.\')\n    datasets = [(Path(root), list_frames(root)) for root in args.data_roots]\n    if len({root.name for root, _ in datasets}) != len(datasets):\n        parser.error(\'Dataset root basenames must be unique.\')\n    if any(not ids for _, ids in datasets):\n        parser.error(\'Every dataset must contain frames.\')\n    # Fail before creating artifacts when CP2 is unfinished.\n    first = load_frame(datasets[0][0], datasets[0][1][0])\n    project(first[\'points\'], first[\'calib\'], first[\'image\'].shape)\n    args.out_dir.mkdir(parents=True, exist_ok=False)\n    out = args.out_dir\n    cv2.setNumThreads(1)\n    env = environment()\n    configs = configurations()\n    manifest = data_manifest(datasets)\n    write_json(out / \'data_manifest.json\', manifest)\n    split_rows = [dict(dataset=root.name, frame=frame, split=split,\n                       scene=frame.rsplit(\'_\', 1)[0] if root.name == \'nuscenes_mini_subset\' else root.name)\n                  for root, ids in datasets for frame, split in split_frames(root.name, ids, args.seed).items()]\n    write_csv(pd.DataFrame(split_rows), out / \'split_manifest.csv\')\n    write_json(out / \'split_manifest.json\', split_rows)\n    env.update(data_manifest=\'data_manifest.json\',\n               data_manifest_sha256=file_sha256(out / \'data_manifest.json\'),\n               split_manifest=\'split_manifest.json\', split_manifest_csv=\'split_manifest.csv\',\n               split_manifest_sha256=file_sha256(out / \'split_manifest.json\'))\n    write_json(out / \'environment.json\', env)\n    write_json(out / \'config.json\', dict(seed=args.seed, roots=[str(r) for r, _ in datasets],\n               frames={r.name: ids for r, ids in datasets}, configurations=configs,\n               sweep_zero=\'One shared zero control included in every factor curve; factors never combined.\',\n               angles_deg=ANGLES, translations_m=TRANSLATIONS, score_samples=args.score_samples,\n               canny=[100, 200], score=\'median candidate-visible edge distance / image diagonal\',\n               min_score_visible=MIN_SCORE_VISIBLE, score_insufficient_status=\'coverage_insufficient\',\n               native_axes=NATIVE_AXES, factor_axes=dict(roll=\'native x\', pitch=\'native y\', yaw=\'native z\',\n                                                       x=\'native x\', y=\'native y\', z=\'native z\'),\n               code_sha256=env[\'code_sha256\'], data_manifest=\'data_manifest.json\',\n               data_manifest_sha256=file_sha256(out / \'data_manifest.json\'),\n               split_manifest=\'split_manifest.json\', split_manifest_csv=\'split_manifest.csv\',\n               split_manifest_sha256=file_sha256(out / \'split_manifest.json\'),\n               min_object_baseline_visible=5, latency_repeats=args.latency_repeats, caveats=CAVEATS))\n    frame_rows, object_rows, catalog, latency_rows, latency_stats = [], [], [], [], []\n    for root, ids in datasets:\n        dataset = root.name\n        splits = split_frames(dataset, ids, args.seed)\n        for index, frame_id in enumerate(ids):\n            frame = load_frame(root, frame_id)\n            points, calib, image = frame[\'points\'], frame[\'calib\'], frame[\'image\']\n            baseline_cam = velo_to_cam(points[:, :3], calib)\n            base_uv, base_valid, _ = project(points, calib, image.shape)\n            assigned = [np.flatnonzero(membership(baseline_cam, obj) & base_valid) for obj in frame[\'labels\']]\n            digest = hashlib.sha256(f\'{args.seed}:{dataset}:{frame_id}:score\'.encode()).digest()\n            rng = np.random.default_rng(int.from_bytes(digest[:8], \'little\'))\n            finite = np.flatnonzero(np.isfinite(points[:, :3]).all(1))\n            sample_ids = np.sort(rng.choice(finite, min(len(finite), args.score_samples), replace=False))\n            distances = edge_map(image)\n            object_distances = [float(np.linalg.norm(obj.location)) for obj in frame[\'labels\']]\n            catalog.append(dict(dataset=dataset, root=str(root), frame=frame_id, split=splits[frame_id],\n                                object_distances_m=object_distances))\n            for cfg in configs:\n                uv, valid, _ = (base_uv, base_valid, None) if cfg[\'config\'] == \'zero\' else project(points, drift(calib, cfg), image.shape)\n                common = base_valid & valid\n                displacement = np.linalg.norm(uv[common] - base_uv[common], axis=1)\n                score, score_visible = edge_score(uv, valid, sample_ids, distances)\n                hits_all, den_all, ratios = 0, 0, []\n                for j, (obj, members) in enumerate(zip(frame[\'labels\'], assigned)):\n                    visible_ids = members[valid[members]]\n                    pixels = uv[visible_ids]\n                    x1, y1, x2, y2 = obj.bbox\n                    hits = int(((pixels[:, 0] >= x1) & (pixels[:, 0] <= x2)\n                                & (pixels[:, 1] >= y1) & (pixels[:, 1] <= y2)).sum())\n                    eligible = len(members) >= 5\n                    ratio = hits / len(members) if eligible else np.nan\n                    if eligible:\n                        hits_all += hits\n                        den_all += len(members)\n                        ratios.append(ratio)\n                    object_rows.append(dict(dataset=dataset, frame=frame_id, split=splits[frame_id], **cfg,\n                        object_id=j, object_type=obj.type, distance_m=object_distances[j], eligible=eligible,\n                        baseline_visible_assigned=len(members), perturbed_visible_assigned=len(visible_ids),\n                        offscreen_misses=len(members) - len(visible_ids), bbox_hits=hits, association_ratio=ratio,\n                        bbox_x1=float(x1), bbox_y1=float(y1), bbox_x2=float(x2), bbox_y2=float(y2)))\n                n, base_n = len(points), int(base_valid.sum())\n                frame_rows.append(dict(dataset=dataset, frame=frame_id, split=splits[frame_id], **cfg,\n                    n_points=n, invalid_xyz=int((~np.isfinite(points[:, :3]).all(1)).sum()),\n                    baseline_inside_count=base_n, inside_count=int(valid.sum()), inside_fov=float(valid.sum() / n) if n else np.nan,\n                    fov_change=float((valid.sum() - base_n) / n) if n else np.nan,\n                    common_valid=int(common.sum()), common_coverage=float(common.sum() / base_n) if base_n else np.nan,\n                    displacement_sum_px=float(displacement.sum()), displacement_mean_px=float(displacement.mean()) if len(displacement) else np.nan,\n                    displacement_median_px=float(np.median(displacement)) if len(displacement) else np.nan,\n                    displacement_p95_px=float(np.percentile(displacement, 95)) if len(displacement) else np.nan,\n                    bbox_hits=hits_all, baseline_visible_assigned=den_all,\n                    association_micro=hits_all / den_all if den_all else np.nan,\n                    association_macro=float(np.mean(ratios)) if ratios else np.nan,\n                    edge_score=score, score_sample_count=len(sample_ids), score_visible_count=score_visible,\n                    score_status=\'ok\' if score_visible >= MIN_SCORE_VISIBLE else \'coverage_insufficient\',\n                    score_coverage=score_visible / len(sample_ids) if len(sample_ids) else np.nan,\n                    sensor_time_delta_ms=(frame[\'timestamp_camera_us\'] - frame[\'timestamp_lidar_us\']) / 1000 if \'timestamp_camera_us\' in frame else np.nan))\n            if index == 0 and dataset in [\'kitti_mini\', \'nuscenes_mini_subset\']:\n                latency_stats.append(latency(frame, dataset, args.latency_repeats, latency_rows))\n            print(f\'{dataset} {index + 1}/{len(ids)} {frame_id}\', flush=True)\n    frames, objects = pd.DataFrame(frame_rows), pd.DataFrame(object_rows)\n    frame_files = write_csv(frames, out / \'per_frame_config.csv\')\n    object_files = write_csv(objects, out / \'per_object_config.csv\')\n    write_json(out / \'csv_manifest.json\', dict(per_frame_config=frame_files,\n               per_object_config=object_files, byte_limit_exclusive=20_000_000,\n               target_shard_rows=20000, aggregation=\'summary uses the entire DataFrame, not a single shard\'))\n    aggregate = summary(frames, objects)\n    write_csv(aggregate, out / \'summary.csv\')\n    write_csv(pd.DataFrame(latency_rows), out / \'latency_repeats.csv\')\n    write_csv(pd.DataFrame([{**row, \'cpu\': env[\'cpu\'], \'platform\': env[\'platform\'], \'opencv_threads\': env[\'opencv_threads\']} for row in latency_stats]), out / \'latency_summary.csv\')\n    evaluation, roc, thresholds = detection(frames, out)\n    plots(aggregate, evaluation, roc, thresholds, out)\n    visual_artifacts(catalog, frames, evaluation, configs, out)\n    write_json(out / \'completion.json\', dict(status=\'complete\', frame_configs=len(frames),\n               object_configs=len(objects), elapsed_scope=\'No training; all listed frames evaluated.\',\n               caveats=CAVEATS))\n    print(f\'Complete: {out}. Read score_evaluation.json and fail_01_facts.json before making claims.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/tests/test_projection.py': '"""Read-only tests: python -B -m unittest discover -s src/tests -v."""\nimport unittest\nfrom unittest.mock import patch\nfrom io import StringIO\n\nimport numpy as np\nimport pandas as pd\n\nfrom starter.kitti_io import load_calib, KittiCalib, KittiObject\nfrom starter.projection import velo_to_cam, cam_to_image\nfrom src.calibration_benchmark import (membership, configurations, split_frames, edge_score,\n                                       overlay, detection_rates, csv_payloads)\n\n\nclass ProjectionTests(unittest.TestCase):\n    def test_known_synthetic_point(self):\n        calib = load_calib(\'data/synthetic/training/calib/000000.txt\')\n        cam = velo_to_cam(np.array([[10., 0., 0.]]), calib)\n        self.assertAlmostEqual(cam[0, 2], 9.72732109, places=6)\n        uv, depth, mask = cam_to_image(cam, calib.P2, (375, 1242, 3))\n        np.testing.assert_allclose(uv, [[613.96414869, 175.00653723]], atol=1e-6)\n        np.testing.assert_allclose(depth, [9.72732109], atol=1e-6)\n        np.testing.assert_array_equal(mask, [True])\n\n    def test_depth_nonfinite_and_image_bounds(self):\n        points = np.array([[1., 1., 1.], [0., 0., 1.], [9.999, 9.999, 1.],\n                           [10., 0., 1.], [0., 10., 1.], [-.01, 0., 1.],\n                           [0., -.01, 1.], [0., 0., -1.], [0., 0., 0.],\n                           [0., 0., .1], [.02, .02, .2],\n                           [np.nan, 0., 1.], [0., np.inf, 1.], [0., 0., np.inf]])\n        P = np.column_stack([np.eye(3), np.zeros(3)])\n        uv, depth, mask = cam_to_image(points, P, (10, 10, 3))\n        np.testing.assert_array_equal(mask, [True, True, True, False, False, False,\n                                           False, False, False, False, True, False, False, False])\n        np.testing.assert_allclose(uv, [[1., 1.], [0., 0.], [9.999, 9.999], [.1, .1]])\n        np.testing.assert_allclose(depth, [1., 1., 1., .2])\n\n    def test_empty_and_all_invalid(self):\n        P = np.column_stack([np.eye(3), np.zeros(3)])\n        for points in [np.empty((0, 3)), np.array([[0., 0., -1.]])]:\n            uv, depth, mask = cam_to_image(points, P, (10, 10))\n            self.assertEqual(uv.shape, (0, 2))\n            self.assertEqual(depth.shape, (0,))\n            self.assertEqual(mask.shape, (len(points),))\n            self.assertFalse(mask.any())\n\n    def test_zero_projective_denominator(self):\n        P = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.], [0., 0., 0., 0.]])\n        uv, depth, mask = cam_to_image(np.array([[0., 0., 1.], [1., 1., 1.]]), P, (10, 10))\n        self.assertFalse(mask.any())\n        self.assertEqual(uv.shape, (0, 2))\n        self.assertEqual(depth.shape, (0,))\n\n    def test_transform_direction(self):\n        transform = np.array([[0., -1., 0., 2.], [1., 0., 0., 3.], [0., 0., 1., 4.]])\n        calib = KittiCalib(np.zeros((3, 4)), np.eye(3), transform)\n        np.testing.assert_allclose(velo_to_cam(np.array([[1., 2., 3.]]), calib), [[0., 4., 7.]])\n\n    def test_calibration_must_be_finite(self):\n        points = np.array([[1., 1., 1.]])\n        for bad in [np.nan, np.inf]:\n            P = np.column_stack([np.eye(3), np.zeros(3)])\n            P[0, 0] = bad\n            with self.assertRaises(ValueError):\n                cam_to_image(points, P, (10, 10))\n            transform = np.column_stack([np.eye(3), np.zeros(3)])\n            transform[0, 0] = bad\n            with np.errstate(invalid=\'ignore\'), self.assertRaises(ValueError):\n                velo_to_cam(points, KittiCalib(np.zeros((3, 4)), np.eye(3), transform))\n            rect = np.eye(3)\n            rect[0, 0] = bad\n            with np.errstate(invalid=\'ignore\'), self.assertRaises(ValueError):\n                velo_to_cam(points, KittiCalib(np.zeros((3, 4)), rect,\n                                             np.column_stack([np.eye(3), np.zeros(3)])))\n\n    def test_image_dimensions_must_be_positive(self):\n        P = np.column_stack([np.eye(3), np.zeros(3)])\n        for shape in [(0, 10), (10, 0), (-1, 10), (10, -1)]:\n            with self.assertRaises(ValueError):\n                cam_to_image(np.array([[1., 1., 1.]]), P, shape)\n\n\nclass BenchmarkContractTests(unittest.TestCase):\n    def test_inverse_yaw_bottom_center(self):\n        obj = KittiObject(\'Car\', 0., 0, 0., np.zeros(4), np.array([2., 2., 4.]),\n                          np.array([3., 5., 10.]), np.pi / 2)\n        local = np.array([[1.9, -1., .9], [2.1, -1., 0.], [0., .01, 0.],\n                          [0., -2.01, 0.], [0., -1., 1.01], [np.nan, 0., 0.]])\n        c, s = np.cos(obj.rotation_y), np.sin(obj.rotation_y)\n        rotation = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])\n        camera = local @ rotation.T + obj.location\n        np.testing.assert_array_equal(membership(camera, obj), [True, False, False, False, False, False])\n\n    def test_independent_sweep_and_deterministic_split(self):\n        configs = configurations()\n        self.assertEqual(len(configs), 43)\n        self.assertEqual(sum(c[\'config\'] == \'zero\' for c in configs), 1)\n        frames = [f\'scene-0103_{i:03d}\' for i in range(40)]\n        first = split_frames(\'nuscenes_mini_subset\', frames, 2026)\n        self.assertEqual(first, split_frames(\'nuscenes_mini_subset\', frames[::-1], 2026))\n        self.assertEqual(sum(v == \'test\' for v in first.values()), 20)\n\n    def test_edge_score_fixed_ids_and_visibility(self):\n        dense = np.repeat([[1., 1.]], 31, axis=0)\n        dense[-1] = np.nan\n        distances = np.arange(16, dtype=float).reshape(4, 4)\n        valid = np.arange(31) < 30\n        score, count = edge_score(dense, valid, np.arange(31), distances)\n        self.assertEqual(count, 30)\n        self.assertAlmostEqual(score, 5 / np.hypot(4, 4))\n        score, count = edge_score(dense, valid, np.arange(29), distances)\n        self.assertEqual(count, 29)\n        self.assertTrue(np.isnan(score))\n\n    def test_recall_includes_unscorable_and_unavailable_threshold(self):\n        group = pd.DataFrame(dict(score_available=[True, True, False], detected=[True, False, False]))\n        rates = detection_rates(group, .1)\n        self.assertAlmostEqual(rates[\'sensitivity_recall\'], 1 / 3)\n        self.assertAlmostEqual(rates[\'sensitivity_recall_conditional\'], .5)\n        self.assertAlmostEqual(rates[\'unscorable_rate\'], 1 / 3)\n        self.assertTrue(rates[\'threshold_available\'])\n        missing = detection_rates(group, None)\n        self.assertIsNone(missing[\'sensitivity_recall\'])\n        self.assertIsNone(missing[\'sensitivity_recall_conditional\'])\n        self.assertFalse(missing[\'threshold_available\'])\n        self.assertTrue(missing[\'unavailable_reason\'])\n\n    def test_csv_shards_byte_limit_and_complete_rows(self):\n        table = pd.DataFrame(dict(id=range(15), text=[\'điểm\' * 10] * 15))\n        single = list(csv_payloads(table, \'objects.csv\'))\n        self.assertEqual(single[0][0], \'objects.csv\')\n        shards = list(csv_payloads(table, \'objects.csv\', max_bytes=300, shard_rows=4))\n        self.assertGreater(len(shards), 1)\n        self.assertEqual(shards[0][0], \'objects_000.csv\')\n        self.assertTrue(all(len(data) < 300 for _, _, data in shards))\n        restored = pd.concat([pd.read_csv(StringIO(data.decode(\'utf-8\'))) for _, _, data in shards], ignore_index=True)\n        pd.testing.assert_frame_equal(restored, table)\n        self.assertEqual(sum(rows for _, rows, _ in shards), len(table))\n\n    def test_overlay_radius_limit_and_selected_object(self):\n        obj = KittiObject(\'Car\', 0., 0, 0., np.array([2., 3., 12., 13.]),\n                          np.array([2., 2., 4.]), np.array([0., 0., 5.]), 0.)\n        frame = dict(image=np.zeros((60, 100, 3), np.uint8), points=np.zeros((12001, 4)),\n                     calib=None, labels=[obj])\n        uv = np.ones((12001, 2)) * 5\n        with patch(\'src.calibration_benchmark.project\', return_value=(uv, np.ones(12001, bool), np.ones(12001))), \\\n             patch(\'src.calibration_benchmark.cv2.circle\') as circles, \\\n             patch(\'src.calibration_benchmark.cv2.rectangle\') as rectangles, \\\n             patch(\'src.calibration_benchmark.cv2.putText\') as texts:\n            image = overlay(frame, configurations()[0], \'test\', selected_object_id=0)\n        self.assertEqual(image.shape, frame[\'image\'].shape)\n        self.assertLessEqual(circles.call_count, 6000)\n        self.assertGreater(circles.call_count, 0)\n        self.assertTrue(all(call.args[2] == 2 for call in circles.call_args_list))\n        self.assertEqual(rectangles.call_args_list[0].args[-1], 3)\n        self.assertIn(\'SELECTED #0 Car 5.00m\', texts.call_args_list[0].args[1])\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
SOURCE_HASHES = {'starter/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'starter/data_health.py': '7ff78a3e4b73c32c153855949125d1defffc0753629f51190a276d3b0e5afbd5', 'starter/datasets.py': 'b24cd72db6fa07073ea5a9caa84988a83142f9e9d9af62f2b1bfc6b30be130d4', 'starter/kitti_io.py': 'b4de0a271a21da12d55691585abbcb2d08b1557bd2974a51709085570e7a84fd', 'starter/nuscenes_io.py': 'f9c5369d672d04578001833c38abaef97559b797316ee7c02fc1f46bd8111b7e', 'starter/perturb.py': '24eb4be500ea78c49f038c0fdf17a9620376641ca8d642ca834e3e12547da9b5', 'starter/projection.py': '40c5e7d157979abf0dbcaef5d1bc80ec418390a0346d8482b06d2f9ff2730211', 'src/__init__.py': 'cc7a57dadb684810a7ebe9cb3acc1f6c6035f8f37051dcfbbe0778a8b0f62e4e', 'src/calibration_benchmark.py': 'aa4ba3b3cbcde0671126be92329c30abbbce238896c4d2828a7ce525c76d4648', 'src/tests/test_projection.py': '60008efaad846b3e923b84c517969e6cd72a9ef3da93b170babed7aa40958f5f'}

for name, source in PAYLOAD.items():
    destination = run_root / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('x', encoding='utf-8') as stream:
        stream.write(source)
    assert hashlib.sha256(destination.read_bytes()).hexdigest() == SOURCE_HASHES[name]
os.chdir(run_root)
sys.path.insert(0, str(run_root))
output = Path('/kaggle/working/results') / ('topic_a_' + run_id)
print('Data revision:', DATA_REVISION)
print('Source hashes verified:', len(SOURCE_HASHES))
print('Fresh output:', output)


In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover('src/tests')
assert suite.countTestCases() > 0, 'Tests must not be silently skipped'
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Required tests failed'


In [ ]:
subprocess.run([sys.executable, '-m', 'src.calibration_benchmark',
                '--out-dir', str(output), '--seed', '2026', '--latency-repeats', '30'], check=True)
completion = json.loads((output / 'completion.json').read_text())
assert completion['status'] == 'complete'
import pandas as pd
metrics = pd.read_csv(output / 'per_frame_config.csv')
expected_frames = {'kitti_mini': 20, 'nuscenes_mini_subset': 80, 'synthetic': 5}
assert metrics.groupby('dataset').frame.nunique().to_dict() == expected_frames
assert len(metrics) == 105 * 43
assert not metrics.duplicated(['dataset', 'frame', 'config']).any()
assert (metrics.groupby(['dataset', 'frame']).config.nunique() == 43).all()
for artifact in ['summary.csv', 'score_evaluation.json', 'fail_01_facts.json', 'latency_summary.csv']:
    assert (output / artifact).is_file(), artifact
with (output / 'kaggle_source_manifest.json').open('x') as stream:
    json.dump({'data_revision': DATA_REVISION, 'source_hashes': SOURCE_HASHES,
               'run_id': run_id, 'dependencies': DEPENDENCIES,
               'test_count': suite.countTestCases()}, stream, indent=2)
print(completion)


## Kết quả và giới hạn

Tỷ lệ điểm đúng box cố định nhóm điểm theo box 3D của lần không gây lệch;
điểm trôi ra ngoài ảnh được tính sai. Độ lệch pixel chỉ đo trên điểm còn nhìn thấy
ở cả hai lần; đọc kèm coverage để tránh che mất điểm rơi khỏi ảnh.

nuScenes sinh box 2D từ nhãn 3D nên kết quả box không phải kiểm chứng độc lập.
Ngưỡng score chỉ lấy từ baseline nhóm calibration; nhóm test không dùng để chỉnh
ngưỡng. Các frame gần nhau tương quan: phép thử cho thấy độ nhạy trong tập này,
chưa xác nhận khả năng phát hiện drift trên xe thật. Không dùng latency máy này
thay cho latency máy địa phương.


In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv(output / 'summary.csv')
display(summary[(summary.config == 'zero') | ((summary.factor == 'yaw') & summary.value.isin([1,2,3]))])
display(pd.read_csv(output / 'latency_summary.csv'))
score_results = json.loads((output / 'score_evaluation.json').read_text())
display(pd.DataFrame(score_results['results']))
failure = json.loads((output / 'fail_01_facts.json').read_text())
print('Failure selected by:', failure['selection_rule'])
print(json.dumps(failure['metrics'], indent=2))
figure_root = output / 'figures' if (output / 'figures').is_dir() else output
for figure in sorted(figure_root.glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure), width=1100))
